# 02 - Preparação dos dados

Roda o ETL de `src/etl/` passo a passo e confere cada saída. O mesmo fluxo roda de uma vez com
`python -m src.etl.run_pipeline`.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

from src.config import MIN_MESES_HISTORICO, MAX_MESES_INTERPOLACAO, HORIZONTE
from src.etl.extract import carregar_bruto
from src.etl.transform import padronizar, validar_linhas, regularizar_grade, resumir_itens, motivo_exclusao_modelagem
from src.features.build_features import construir_base, linhas_utilizaveis, FEATURES_NUMERICAS

## 1. Padronização e validação

In [ ]:
df = padronizar(carregar_bruto())
df, rejeitadas = validar_linhas(df)
print("Linhas válidas:", len(df), "| rejeitadas:", len(rejeitadas))
df.head(3)

In [ ]:
# conferindo as correções de categoria
df.loc[df["categoria"] != df["categoria_original"].map({
    "Bakery and grains": "Padaria e grãos", "Beef": "Carne bovina", "Dairy and fats": "Laticínios",
    "Drinks": "Bebidas", "Eggs": "Ovos", "Energy": "Energia", "Fruit": "Frutas",
    "Pantry and snacks": "Mercearia", "Pork and deli": "Suínos e frios", "Poultry": "Aves", "Vegetables": "Hortaliças",
}), ["item", "categoria_original", "categoria"]].drop_duplicates()

## 2. Grade mensal

Cada série vira uma sequência mensal contínua do primeiro ao último mês publicado. Buracos de até
2 meses são interpolados em escala log (a média geométrica respeita melhor variações percentuais);
maiores ficam vazios.

In [ ]:
precos = regularizar_grade(df)
precos["imputado"].groupby(precos["data"]).sum().loc[lambda s: s > 0].sort_values(ascending=False).head(8)

In [ ]:
# exemplo: ovos em torno do shutdown
precos[(precos["item"].str.startswith("Eggs")) & (precos["data"].between("2025-08-01", "2026-01-01"))][["data", "preco", "imputado"]]

## 3. Tabela de itens e filtro da modelagem

In [ ]:
itens = resumir_itens(precos)
itens["motivo_exclusao"] = motivo_exclusao_modelagem(itens)
print("Na modelagem:", itens["motivo_exclusao"].isna().sum(), "de", len(itens))
itens[itens["motivo_exclusao"].notna()][["item", "fim", "meses_publicados", "meses_faltando", "motivo_exclusao"]].sort_values("motivo_exclusao")

## 4. Base de modelagem

Uma linha por (série, mês-base t). O alvo é `log(preço em t+3) - log(preço em t)`.

| Feature | O que é |
|---|---|
| `ret_1`, `ret_3`, `ret_6`, `ret_12` | variação (log) nos últimos 1, 3, 6 e 12 meses |
| `dist_media_12` | quão acima/abaixo da média dos últimos 12 meses o preço está |
| `vol_6` | desvio-padrão das variações mensais nos últimos 6 meses |
| `sazonal_ano_anterior` | variação que aconteceu na mesma janela do ano anterior |
| `ret_3_categoria` | variação média de 3 meses da categoria no mesmo mês |
| `gasolina_ret_3`, `gasolina_ret_12` | variação recente da gasolina comum |
| `mes_alvo_sin`, `mes_alvo_cos` | mês do alvo em forma cíclica |
| `categoria` | one-hot |

In [ ]:
series_ok = itens.loc[itens["motivo_exclusao"].isna(), "serie_id"]
base = construir_base(precos, series_ok)
uteis = linhas_utilizaveis(base)
print("Linhas na base:", len(base), "| com features e alvo completos:", len(uteis))
uteis[["item", "data_base", "data_alvo", "preco", "preco_alvo", "alvo"] + FEATURES_NUMERICAS[:4]].head()

In [ ]:
# por que tantas linhas caem? principalmente os 12 primeiros meses de cada série (sem ret_12)
# e os 3 últimos (alvo ainda não existe)
base[FEATURES_NUMERICAS + ["alvo"]].isna().sum().sort_values(ascending=False).head(6)

## 5. Validações antes de salvar

In [ ]:
assert base.duplicated(["serie_id", "data_base"]).sum() == 0
assert (uteis["preco_alvo"] > 0).all()
assert np.allclose(uteis["alvo"], np.log(uteis["preco_alvo"] / uteis["preco"]))
assert not uteis["alvo_imputado"].any(), "alvo interpolado não pode entrar em treino/teste"
print("ok")

In [ ]:
from src.etl import run_pipeline
run_pipeline.main()